# 300. Longest Increasing Subsequence
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/longest-increasing-subsequence/

## 💡 Concepts

**Core concept(s):** DP where `dp[i]` = longest increasing run **ending at** `i`; or a clever **binary-search** "patience" method.

**Why it applies here:** The best run ending at `i` extends the best earlier run that ends in a smaller value. That's `O(n²)`. Faster: keep a list of the smallest possible tail for each run-length; binary-search where each number fits — the list's length is the answer.

**Key intuition:** Each number extends the best compatible earlier run; or slot each number into a 'smallest tails' list.

---

### 📚 What is Dynamic Programming (DP)?
**DP** solves a big problem by solving smaller **overlapping** subproblems once and reusing the answers. Two styles: **memoization** (recursion that caches results) and **tabulation** (fill a table from the smallest cases up).
- **Why it's fast:** it turns exponential re-computation into a single sweep over the subproblems.
- **In Python:** a `dict`/list cache, or a `dp` list/2-D table.

### 📚 What is Binary Search (used here)?
**Binary search** finds a position in a **sorted** list in **O(log n)** by halving the range each step. Here it slots each number into a running "best tails" list, giving an `O(n log n)` solution.

---

**Prerequisite knowledge:**
- 1-D DP over 'ending at i'.
- Binary search (`bisect`).

## 📝 Problem

Return the length of the longest strictly increasing subsequence (not necessarily contiguous).

**Example**
```
[10,9,2,5,3,7,101,18] -> 4   (2,3,7,101)
```

> Two approaches: DP `O(n²)` and binary-search `O(n log n)`.

### Approach 1 — DP, "longest ending here" (worst)

**Idea:** `dp[i]` = 1 + the best `dp[j]` for `j < i` with `nums[j] < nums[i]`. Answer is the max.

**Time:** `O(n²)`. **Space:** `O(n)`.

In [ ]:
def lis_dp(nums):
    if not nums:
        return 0
    dp = [1] * len(nums)                   # dp[i] = length of the best increasing run ENDING at i
    for i in range(len(nums)):
        for j in range(i):                 # look at every earlier element
            if nums[j] < nums[i]:          # can we extend that run with nums[i]?
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)                         # the overall best run

### Approach 2 — Patience + Binary Search (optimal)

**Idea:** Keep `tails`, where `tails[k]` is the smallest possible tail of an increasing run of length `k+1`. For each number, binary-search its slot: it either extends `tails` or replaces the first tail ≥ it. The length of `tails` is the answer.

**Time:** `O(n log n)`. **Space:** `O(n)`.

In [ ]:
import bisect

def lis_binary(nums):
    tails = []                             # tails[k] = smallest possible tail of a run of length k+1
    for x in nums:
        i = bisect.bisect_left(tails, x)   # first tail >= x
        if i == len(tails):
            tails.append(x)                # x extends the longest run so far
        else:
            tails[i] = x                   # x is a smaller tail for that run length
    return len(tails)                      # number of "run lengths" we could build = the answer

In [ ]:
# Correctness check
tests = [([10,9,2,5,3,7,101,18],4), ([0,1,0,3,2,3],4), ([7,7,7,7],1), ([],0)]
for nums, exp in tests:
    a, b = lis_dp(nums), lis_binary(nums)
    print(f"{nums} -> dp={a}, binary={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

Inputs are shaped to force the worst case. (Exponential brute-force versions are shown in the code but omitted from timing where they would blow up — noted per notebook.)

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (list(range(n)),)   # strictly increasing -> DP does full inner loops
solutions = {
    "dp     O(n^2)    ": lis_dp,
    "binary O(n log n)": lis_binary,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **"Best ending here" DP:** a workhorse for subsequence problems.
- **Patience sorting:** maintaining smallest tails + binary search cuts `O(n²)` to `O(n log n)`.
- **Signal:** "longest increasing/chain subsequence", "can this be sped up past n²".
- **Related problems:** Russian Doll Envelopes, Number of LIS, Longest Chain of Pairs.
- **Common pitfalls:** (1) confusing subsequence with subarray; (2) `bisect_left` vs `bisect_right` (strict vs non-strict).